## 2.1 Valores nulos

¿Hay datos faltantes? ¿Siguen algún patrón?

In [ ]:
# Mapa de nulos
nulos = df.isnull().sum()
nulos_pct = (df.isnull().sum() / len(df) * 100).round(1)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    '% Nulos': nulos_pct,
    'Tipo': df.dtypes
}).sort_values('Nulos', ascending=False)

print(f"Columnas sin nulos: {(nulos == 0).sum()}/{len(df.columns)}")
print(f"Columnas con nulos: {(nulos > 0).sum()}/{len(df.columns)}\n")
resumen_nulos[resumen_nulos['Nulos'] > 0]

**Interpretación de nulos:**
- Los datos de Ember (mensuales: generación, demanda, CO₂) están **100% completos** — excelente
- Los nulos están en columnas **anuales de OWID** (población, PIB, shares) — esto es normal porque OWID publica con retraso de 1-2 años
- **No necesitamos imputar**: las columnas anuales son contexto, no features críticas. El modelo usará las columnas mensuales

## 2.2 Consistencia temporal

¿Tenemos todos los meses sin saltos?

In [ ]:
# Verificar continuidad temporal
df_sorted = df.sort_values('fecha')
diffs = df_sorted['fecha'].diff().dt.days.dropna()

# ¿Hay saltos > 35 días (más de un mes)?
saltos = diffs[diffs > 35]

rango_esperado = pd.date_range(df['fecha'].min(), df['fecha'].max(), freq='MS')
meses_faltantes = set(rango_esperado) - set(df['fecha'])

print(f"Rango: {df['fecha'].min().strftime('%Y-%m')} → {df['fecha'].max().strftime('%Y-%m')}")
print(f"Meses en dataset: {len(df)}")
print(f"Meses esperados: {len(rango_esperado)}")
print(f"Meses faltantes: {len(meses_faltantes)}")
print(f"Duplicados de fecha: {df['fecha'].duplicated().sum()}")
print(f"\nDiferencia entre meses consecutivos (días):")
print(f"  Min: {diffs.min():.0f}, Max: {diffs.max():.0f}, Media: {diffs.mean():.1f}")

if len(meses_faltantes) > 0:
    print(f"\nMeses faltantes: {sorted(meses_faltantes)}")
else:
    print("\n✓ Serie temporal completa, sin saltos")

## 2.3 Outliers estadísticos

Usamos el método IQR (rango intercuartil) para identificar valores extremos. 

**Importante:** En nuestro caso, los outliers estadísticos podrían ser **anomalías reales** (crisis energéticas), no errores de datos. No los eliminamos — los marcamos.

In [ ]:
# Detección de outliers con IQR
cols_analisis = ['demanda_twh', 'gen_hydro', 'gen_fossil', 'gen_gas', 
                 'co2_intensity_gco2_kwh', 'importaciones_netas_twh']
cols_analisis = [c for c in cols_analisis if c in df.columns]

outlier_report = []
for col in cols_analisis:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    
    outlier_report.append({
        'Variable': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Límite inferior': round(lower, 2),
        'Límite superior': round(upper, 2),
        'N° outliers': len(outliers),
        '% outliers': f"{len(outliers)/len(df)*100:.1f}%"
    })
    
    if len(outliers) > 0:
        print(f"\n{col}: {len(outliers)} outliers")
        for _, row in outliers.iterrows():
            print(f"  {row['fecha'].strftime('%Y-%m')}: {row[col]:.2f} (rango normal: [{lower:.2f}, {upper:.2f}])")

pd.DataFrame(outlier_report)

In [ ]:
# Boxplots de variables principales
fig = px.box(df.melt(id_vars='fecha', value_vars=cols_analisis),
             x='variable', y='value', color='variable',
             title='Distribución de Variables Principales — Boxplot',
             labels={'value': 'Valor', 'variable': ''})
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

## 2.4 Validación de consistencia

Verificamos que los datos tienen sentido físico:
- `gen_hydro + gen_fossil ≈ gen_total` (las fuentes suman al total)
- La demanda no puede ser negativa
- Las importaciones netas pueden ser positivas (importa) o negativas (exporta)

In [ ]:
# Verificar que las partes suman ~100% (generación por fuente)
if 'gen_clean' in df.columns and 'gen_fossil' in df.columns:
    df['_check_sum'] = df['gen_clean'] + df['gen_fossil']
    print("Verificación: gen_clean + gen_fossil debería ≈ 100%")
    print(f"  Media: {df['_check_sum'].mean():.1f}%")
    print(f"  Min: {df['_check_sum'].min():.1f}%, Max: {df['_check_sum'].max():.1f}%")
    df.drop('_check_sum', axis=1, inplace=True)

# Verificar valores negativos donde no deberían existir
print("\nValores negativos:")
for col in ['demanda_twh', 'gen_hydro', 'gen_fossil', 'gen_total_generation']:
    if col in df.columns:
        n_neg = (df[col] < 0).sum()
        print(f"  {col}: {n_neg} negativos {'✓' if n_neg == 0 else '⚠️'}")

# Importaciones netas sí pueden ser negativas (exportación)
if 'importaciones_netas_twh' in df.columns:
    n_export = (df['importaciones_netas_twh'] < 0).sum()
    n_import = (df['importaciones_netas_twh'] > 0).sum()
    print(f"\n  Meses exportando: {n_export}")
    print(f"  Meses importando: {n_import}")

## 2.5 Conclusiones de calidad

| Aspecto | Estado | Acción |
|---------|--------|--------|
| Datos mensuales Ember | 100% completos | Ninguna |
| Datos anuales OWID | ~15% nulos (años recientes) | No imputar, son contexto |
| Continuidad temporal | Sin saltos | Ninguna |
| Duplicados | 0 | Ninguna |
| Outliers IQR | Coinciden con crisis 2024 | **No eliminar** — son anomalías reales |
| Consistencia física | Fuentes suman correctamente | Ninguna |
| Valores negativos | Solo en importaciones (correcto) | Ninguna |

**Veredicto:** Los datos son de **alta calidad**. No necesitan limpieza significativa. Los "outliers" son eventos reales del sector eléctrico ecuatoriano que queremos detectar, no errores.

---
*Siguiente notebook: 03 — Análisis de patrones*

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/ecuador_electricity_real.csv', parse_dates=['fecha'])

# 02 — Limpieza y Calidad de Datos

## Objetivo
Evaluar la calidad de los datos, identificar valores faltantes, outliers estadísticos, y preparar un dataset limpio para análisis.

**Puntos a verificar:**
- Valores nulos y su patrón
- Consistencia temporal (¿faltan meses?)
- Outliers estadísticos vs anomalías reales
- Tipos de datos correctos
- Duplicados